### Momentum Feature

In [21]:
import pandas as pd
import sqlite3
from sector_rotation.config import SECTORS

# Open a connection to your database file
con = sqlite3.connect("../data/sector_rotation.db")

# Build the "?" placeholders for the SQL query — one per ticker
placeholders = ",".join("?" * len(SECTORS))

# The SQL query. In plain English:
#   "Give me the date, ticker, value columns FROM the prices table,
#    but ONLY rows where ticker is one of my 11,
#    sorted by date then ticker."
# The {placeholders} slots in that "?,?,?..." string via an f-string.
q = f"SELECT date, ticker, value FROM prices WHERE ticker IN ({placeholders}) ORDER BY date, ticker"

# Load into a df
long = pd.read_sql(q, con, params=SECTORS, parse_dates=["date"])

# Close the connection
con.close()

# ENSURE PROPER LOADING INTO DF
# print(long.shape)
# long["ticker"].value_counts()

# REORGANIZE THE DF SO THE DATA IS THE INDEX, ALL THE TICKERS ARE COLUMNS AND THE VALUES IN ARE PRICE VALUES
wide = long.pivot(index="date", columns="ticker", values="value")
wide = wide[SECTORS]

# ENSURE THE ABOVE CODE WORKS
# print(wide.shape)
# print(wide.isna().sum())
# wide.tail()
print(wide.index.dtype)

# RESAMPLE THE DATA SO THAT WE ONLY HAVE ONE PIECE OF DATA PER WEEK INSTEAD OF DAILY
weekly = wide.resample("W-FRI").last()

# ENSURE ABOVE CODE WORKS
# print(weekly.shape)
# print(weekly.index[-5:])
# weekly.tail()

mom_4 = weekly.pct_change(4, fill_method=None)
mom_12 = weekly.pct_change(12, fill_method=None)
mom_26 = weekly.pct_change(26, fill_method=None)

print(mom_4[["XLK", "XLRE", "XLC"]].tail())
print(mom_4["XLC"].notna().idxmax())
print(mom_4.loc["2018-07-06":"2018-07-27", ["XLC","XLK"]])

datetime64[us]
ticker           XLK      XLRE       XLC
date                                    
2026-06-19  0.061256 -0.015709 -0.052053
2026-06-26 -0.050755  0.037320 -0.079765
2026-07-03  0.002796  0.008207 -0.015931
2026-07-10  0.006495 -0.011577  0.002566
2026-07-17 -0.050379  0.024757  0.038657
2018-07-20 00:00:00
ticker           XLC       XLK
date                          
2018-07-06       NaN -0.005121
2018-07-13       NaN  0.011433
2018-07-20  0.007526  0.024718
2018-07-27 -0.012518  0.034259
